# ML Assignment 2 – Classification on the Bank Marketing Dataset

**Programme:** M.Tech (AIML/DSE) – Machine Learning

**Notebook flow:** 1) Dataset → 2) Preprocessing → 3) Models & metrics → 4) Visualisation → 5) Observations

> Run all cells top to bottom (`Kernel → Restart & Run All`) on BITS Virtual Lab and take your screenshot from that run.

## ✅ Submission audit – TODO checklist (delete this cell before final submit)

**Deadline: 27-Nov-2026 23:59. Single PDF, no drafts.**

**Already satisfied (verified from your run)**

- [x] Dataset ≥ 12 features (16) and ≥ 1,000 rows (45,211); UCI source
- [x] Missing values identified (`unknown`) and handled; duplicates checked (0 found)
- [x] Categorical encoding (one-hot) + scaling; 80:20 stratified split; `random_state=42`
- [x] Preprocessing inside `Pipeline` → fitted on training data only
- [x] 5 required models on the same data; Accuracy, AUC, Precision, Recall, F1, MCC all computed
- [x] Metric graphs, confusion matrices, ROC curves; `test_data.csv`, `results.csv`, saved models

**TODO – you must do these**

- [ ] **TODO 1** – Write *Why this dataset* (Section 1)
- [ ] **TODO 2** – Write class-distribution observation (Section 1; 88.3 % / 11.7 %, 7.55:1)
- [ ] **TODO 3** – Write the 5 model observations + overall winner (Section 5)
- [ ] **TODO 4** – Re-run **Restart & Run All on BITS Virtual Lab** and take the screenshot (your run shows `C:\Users\470188\files`, i.e. a local machine)
- [ ] **TODO 5** – Decide 5 vs 6 models (brief lists 5 but one line says 6); make `README.md`, `assignment.py` and notebook consistent
- [ ] **TODO 6** – Complete README: problem statement, feature types, missing values, class distribution, GitHub link, observations
- [ ] **TODO 7** – Rename `_gitignore` → `.gitignore`; commit `data/bank-full.csv`, `test_data.csv`, `model/`, `outputs/`, `requirements.txt`, `README.md`
- [ ] **TODO 8** – Make several meaningful commits (history is reviewed for plagiarism)
- [ ] **TODO 9** – Build ONE PDF in order: GitHub link → README content → BITS Virtual Lab screenshot

## 0. Setup

In [ ]:
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score,
                             f1_score, matthews_corrcoef, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42   # fixed seed for reproducibility
TEST_SIZE = 0.20    # 80:20 split

# Works whether the notebook sits in the repo root or inside model/
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "model" else cwd
DATA_PATH = PROJECT_ROOT / "data" / "bank-full.csv"
MODEL_DIR = PROJECT_ROOT / "model"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
print("Project root:", PROJECT_ROOT)

## 1. Dataset choice

**Dataset:** Bank Marketing (`bank-full.csv`), UCI Machine Learning Repository – <https://archive.ics.uci.edu/dataset/222/bank+marketing>

**Why this dataset:** **TODO 1** – write 2–3 sentences in your own words (e.g. real-world business problem, mix of numeric and categorical features, natural class imbalance, genuine "unknown" values to handle).

**Target variable:** `y` – whether the client subscribed to a term deposit (`yes` / `no`).

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Place bank-full.csv at {DATA_PATH} (download once from UCI and commit it to the repo)."
    )

df = pd.read_csv(DATA_PATH, sep=";")
print("Shape (rows, columns):", df.shape)
print(f"Features (excluding target): {df.shape[1] - 1}")
df.head()

In [ ]:
numeric_cols = df.drop(columns="y").select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.drop(columns="y").select_dtypes(exclude=np.number).columns.tolist()
print(f"Numeric features ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")
df.dtypes.to_frame("dtype")

### Class distribution (imbalance check)

In [ ]:
counts = df["y"].value_counts()
pct = df["y"].value_counts(normalize=True) * 100
print(pd.DataFrame({"count": counts, "percent": pct.round(2)}))
print(f"\nImbalance ratio (no : yes) = {counts['no'] / counts['yes']:.2f} : 1")

fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(counts.index, counts.values, color=["tab:blue", "tab:orange"])
ax.set_title("Target distribution"); ax.set_ylabel("Count")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "class_distribution.png", dpi=150); plt.show()

**TODO 2 – Observation (write in your own words):** state the yes/no percentages above and explain why accuracy alone is misleading here (always predicting "no" already scores about 88 %), so Precision, Recall, F1, AUC and MCC matter.

## 2. Data preprocessing

### 2.1 Missing values

In this dataset missing values are encoded as the text `"unknown"`, so `isna()` finds none. Meaning differs per column:

- `job`, `education` → genuinely missing → convert to `NaN` and impute (inside the pipeline).
- `contact`, `poutcome` → `"unknown"` is informative (e.g. no previous campaign outcome) → keep as its own category.
- `pdays = -1` means the client was **not previously contacted** (a sentinel, not a real number of days).

In [ ]:
print("isna() count (misleading):", int(df.isna().sum().sum()))

unknown_counts = (df.drop(columns="y") == "unknown").sum()
unknown_counts = unknown_counts[unknown_counts > 0]
print("\n'unknown' values per column:")
print(pd.DataFrame({"count": unknown_counts, "percent": (unknown_counts / len(df) * 100).round(2)}))

print("\npdays == -1 (never contacted before):", int((df["pdays"] == -1).sum()),
      f"({(df['pdays'] == -1).mean() * 100:.1f}%)")

In [ ]:
TRUE_MISSING_COLS = ["job", "education"]
df[TRUE_MISSING_COLS] = df[TRUE_MISSING_COLS].replace("unknown", np.nan)
print(df[TRUE_MISSING_COLS].isna().sum())

### 2.2 Duplicates

In [ ]:
n_dup = int(df.duplicated().sum())
print("Exact duplicate rows:", n_dup)
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)

### 2.3 Feature engineering and leakage control

- `duration` (last call length) is only known **after** the call, so it would leak the outcome into a pre-call prediction. It is dropped.
- `pdays = -1` is replaced by a binary flag `previously_contacted`, and `pdays` becomes 0 for those clients so the sentinel does not distort scaling.

In [ ]:
df["previously_contacted"] = (df["pdays"] != -1).astype(int)
df["pdays"] = df["pdays"].replace(-1, 0)

X = df.drop(columns=["y", "duration"])
y = df["y"].map({"no": 0, "yes": 1}).astype(int)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
print(f"Numeric ({len(numeric_features)}): {numeric_features}")
print(f"Categorical ({len(categorical_features)}): {categorical_features}")

### 2.4 Stratified 80:20 train-test split (fixed seed)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Positive rate – train: %.4f | test: %.4f" % (y_train.mean(), y_test.mean()))

# Save test data required by the brief
test_data = X_test.copy()
test_data["y"] = y_test.map({0: "no", 1: "yes"})
test_data.to_csv(PROJECT_ROOT / "test_data.csv", index=False)
print("Saved test_data.csv with", len(test_data), "rows")

### 2.5 Encoding, imputation and scaling (fitted on training data only)

All transformers live inside a `Pipeline`, so `fit` sees only `X_train`; the test set is only transformed.

- Numeric: median imputation (+ `StandardScaler` for models sensitive to scale: Logistic Regression, kNN, Naive Bayes).
- Categorical: most-frequent imputation + one-hot encoding (`handle_unknown="ignore"`).
- Tree-based models do not need scaling.

In [ ]:
def make_preprocessor(scale_numeric):
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))
    cat_steps = [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
    return ColumnTransformer([
        ("num", Pipeline(num_steps), numeric_features),
        ("cat", Pipeline(cat_steps), categorical_features),
    ])

## 3. Models and evaluation metrics

In [ ]:
# (classifier, needs_scaling)
model_specs = {
    "Logistic Regression": (LogisticRegression(class_weight="balanced", max_iter=1000,
                                               random_state=RANDOM_STATE), True),
    "Decision Tree": (DecisionTreeClassifier(class_weight="balanced", max_depth=10,
                                             min_samples_leaf=5, random_state=RANDOM_STATE), False),
    "kNN": (KNeighborsClassifier(n_neighbors=7, weights="distance"), True),
    "Naive Bayes": (GaussianNB(), True),
    "Random Forest (Ensemble)": (RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                                        min_samples_leaf=2, n_jobs=-1,
                                                        random_state=RANDOM_STATE), False),
}

models = {name: Pipeline([("preprocessor", make_preprocessor(scale)), ("classifier", clf)])
          for name, (clf, scale) in model_specs.items()}
list(models)

> **TODO 6a** – In the README, document these configurations and *why* (class_weight for the 7.55:1 imbalance; max_depth/min_samples_leaf to limit overfitting; k=7 with distance weights; scaling only for LR/kNN/NB).

In [ ]:
rows, fitted, preds, scores = [], {}, {}, {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    fitted[name], preds[name], scores[name] = pipe, pred, proba
    rows.append({
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, pred),
        "AUC": roc_auc_score(y_test, proba),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "MCC": matthews_corrcoef(y_test, pred),
    })
    joblib.dump(pipe, MODEL_DIR / (name.split(" (")[0].lower().replace(" ", "_") + ".joblib"))
    print("Trained and saved:", name)

results = pd.DataFrame(rows).set_index("ML Model Name")
results.to_csv(OUTPUT_DIR / "results.csv", float_format="%.4f")
results.round(4)

> **TODO 5 / 6b** – Copy this measured table into the README. The brief mentions "6 models" once but lists 5 required models; this notebook has 5. Keep 5, or add a 6th here **and** in the README – but keep everything consistent.

## 4. Visualisation

In [ ]:
metrics = ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
colors = plt.get_cmap("tab10").colors
for ax, m in zip(axes.flat, metrics):
    bars = ax.bar(results.index, results[m], color=colors[:len(results)])
    ax.bar_label(bars, fmt="%.3f", fontsize=8)
    ax.set_title(m); ax.set_ylim(-0.1 if m == "MCC" else 0, 1)
    ax.tick_params(axis="x", rotation=30); ax.grid(axis="y", alpha=0.3)
fig.suptitle("Model comparison on the 20% stratified test set", fontsize=14)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "metric_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(fitted), figsize=(4 * len(fitted), 4))
for ax, (name, pred) in zip(axes, preds.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["no", "yes"],
                                            colorbar=False, ax=ax)
    ax.set_title(name, fontsize=9)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "confusion_matrices.png", dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, s in scores.items():
    RocCurveDisplay.from_predictions(y_test, s, name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_title("ROC curves"); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curves.png", dpi=150); plt.show()

## 5. Observations

Write these **in your own words** using the measured table and graphs above (the brief checks for plagiarism and identical outputs/text).

| ML Model Name | Observation about model performance |
| --- | --- |
| Logistic Regression | **TODO 3** |
| Decision Tree | **TODO 3** |
| kNN | **TODO 3** |
| Naive Bayes | **TODO 3** |
| Random Forest (Ensemble) | **TODO 3** |
| **Overall winner for your dataset?** | **TODO 3** – justify with AUC, F1 and MCC, not accuracy alone |

> Hints for TODO 3 (not the answer): for each model, comment on the **precision vs recall trade-off**, how **accuracy disagrees with AUC/MCC**, and what the confusion matrices show. Use the rank table below.

In [ ]:
# Helper only: rank models per metric to guide your own observations
print(results.rank(ascending=False).astype(int))
print("\nHighest MCC:", results["MCC"].idxmax(), "| Highest AUC:", results["AUC"].idxmax())

## 6. Final hand-in TODO

- [ ] **TODO 4** Screenshot of this notebook executed on BITS Virtual Lab (Restart & Run All)
- [ ] **TODO 6** README completed, same content pasted in the PDF
- [ ] **TODO 7/8** GitHub repo link opens publicly; commit history shows incremental work
- [ ] **TODO 9** PDF order: GitHub link → README → screenshot